# 🏮 Game Localisation Studio v0.3
### Direct a team of AI agents to localise a Hong Kong mystery game into English

**What you will do (Mission 1 takes about 80 minutes, including setup):** configure a **translator agent** and a **reviewer agent**, run them, **change the translator's prompt and see how the translation changes**, compare runs side by side, judge every reviewer suggestion, direct a revision, approve the final text — and then **play your translation in the browser game**.

> **You are the localisation director. The agents are collaborators, not authorities.**

**Work in pairs, with two roles** (you will swap during the session):

| Role | What you do |
|---|---|
| 🎬 **AI Director** | configures the agents, decides which changes are made, approves the final translation |
| 🔍 **Quality Evaluator** | challenges the outputs, finds errors, and justifies better alternatives |

**How this notebook works:** you never edit Python. Click ▶ on each grey cell (or *Runtime → Run all*), then use the buttons and boxes that appear. Your work is **saved automatically** after every action. If Colab disconnects, run Step 0 again and click **Resume my saved session**. The autosave lives in this Colab runtime only, so click **Download progress** before every break: if Colab resets the runtime, **Resume from file** brings your work back.

| Section | Time (Mission 1) |
|---|---|
| 0 · Setup | ≈3 min |
| 1 · Connect | ≈5 min |
| 2 · Choose your mission | ≈5 min |
| 3 · Configure your agents | ≈15 min |
| 4 · Run the agents | ≈20 min |
| 5 · Human Director | ≈25 min |
| 6 · Test and export | ≈10 min |
| **Total** | **≈83 min** |

**Advanced missions (2–4):** ≈ 35–40 min each. You are already connected, so skip Section 1 and start at Section 2.


In [ ]:
#@title ⚙️ Step 0 · Set up the studio — click ▶ and wait (about 20 seconds) { display-mode: "form" }
REPO_URL = "https://github.com/enricoconductive/game-localisation-studio"
BUNDLE_NAME = "studio_bundle.zip"
COLAB_TARGET = "/content/game-localisation-studio"
MAX_BUNDLE_BYTES = 40_000_000        # size of the uploaded zip
MAX_UNPACKED_BYTES = 80_000_000      # total size after unpacking
MAX_BUNDLE_FILES = 3000

import html, io, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _safe_extract(data, dest):
    """Unpack the bundle zip safely: no absolute paths, no '..', no links, size caps."""
    if len(data) > MAX_BUNDLE_BYTES:
        raise ValueError("The file is too large to be the studio bundle.")
    try:
        zf = zipfile.ZipFile(io.BytesIO(data))
    except zipfile.BadZipFile:
        raise ValueError("This is not a zip file. Upload studio_bundle.zip from your instructor.")
    infos = zf.infolist()
    if len(infos) > MAX_BUNDLE_FILES:
        raise ValueError("The zip has too many files to be the studio bundle.")
    total = 0
    dest = Path(dest).resolve()
    for info in infos:
        name = info.filename.replace("\\", "/")
        parts = PurePosixPath(name).parts
        if (name.startswith("/") or (parts and ":" in parts[0]) or ".." in parts
                or (info.external_attr >> 16) & 0o170000 == 0o120000):
            raise ValueError(f"Unsafe entry in the zip ({name!r}). Ask your instructor for a new bundle.")
        total += info.file_size
        if total > MAX_UNPACKED_BYTES:
            raise ValueError("The zip unpacks to more data than the studio bundle should.")
    for info in infos:
        target = (dest / info.filename.replace("\\", "/")).resolve()
        if dest != target and dest not in target.parents:
            raise ValueError("Unsafe entry in the zip. Ask your instructor for a new bundle.")
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info) as src, open(target, "wb") as out:
            shutil.copyfileobj(src, out)
    return dest


def _find_root(base):
    base = Path(base)
    for cand in [base] + [p for p in base.iterdir() if p.is_dir()]:
        if (cand / "src" / "studio_ui.py").is_file():
            return cand
    return None


def _install_bundle(data, target):
    tmp = Path(tempfile.mkdtemp(prefix="studio_bundle_"))
    try:
        _safe_extract(data, tmp)
        root = _find_root(tmp)
        if root is None:
            raise ValueError("This zip does not contain the studio (src/studio_ui.py is missing).")
        target = Path(target)
        if target.exists():
            shutil.rmtree(target)
        shutil.move(str(root), str(target))
        return target
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def _boot(root):
    root = str(Path(root).resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    os.environ.setdefault("STUDIO_REPO_ROOT", root)
    from src.studio_ui import Studio
    return Studio.boot(repo_root=root)


class _NotReady:
    """Stands in for the studio until setup has finished."""
    def __getattr__(self, name):
        def _explain(*args, **kwargs):
            print("⏳ The studio is not set up yet. Scroll up to Step 0 and finish it "
                  "(upload studio_bundle.zip if it asks for it), then run this cell again.")
        return _explain


def _ask_for_bundle(target):
    import ipywidgets as widgets
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass
    upload = widgets.FileUpload(accept=".zip", multiple=False, description="Upload bundle")
    status = widgets.HTML("<b>The studio could not be downloaded automatically</b> (the course page "
                          "may be private or offline). Upload <code>studio_bundle.zip</code> from your "
                          "instructor with the button below, or drag it into the <b>Files</b> panel on the left "
                          "and run Step 0 again.")

    def _files(value):
        if isinstance(value, dict):  # ipywidgets 7
            return [(k, bytes(v.get("content", b""))) for k, v in value.items()]
        return [(v.get("name", "upload"), bytes(v.get("content", b""))) for v in (value or ())]

    def _on_upload(change):
        global studio
        files = _files(upload.value)
        if not files:
            return
        status.value = "Unpacking…"
        try:
            root = _install_bundle(files[0][1], target)
            studio = _boot(root)
            status.value = ("✅ <b>Done.</b> The studio is installed. Continue with Section 1 below.")
        except Exception as e:  # noqa: BLE001
            status.value = "❌ " + html.escape(f"{type(e).__name__}: {e}")   # never render error text as HTML

    upload.observe(_on_upload, names="value")
    display(widgets.VBox([status, upload]))


def _update_copy(target):
    """Rerunning Step 0: fetch the instructor's latest fixes; keep the old copy if that fails."""
    try:
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        # fetch + reset (not pull): always lands on the latest course version of the code;
        # student work lives outside this folder (/content/studio_state.json, /content/exports)
        subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin", "main"],
                       check=True, capture_output=True, timeout=30, env=env)
        subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                       check=True, capture_output=True, timeout=30, env=env)
        print("Updated the studio to the latest version.")
    except Exception:
        print("Using the copy already downloaded.")


def _setup():
    global studio
    studio = _NotReady()
    env_root = os.environ.get("STUDIO_REPO_ROOT")
    if env_root and (Path(env_root) / "src" / "studio_ui.py").is_file():
        studio = _boot(env_root)
        return
    if _in_colab():
        target = Path(COLAB_TARGET)
        if (target / "src" / "studio_ui.py").is_file():
            _update_copy(target)
        else:
            print("Downloading the studio…")
            try:
                subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True,
                               capture_output=True, timeout=120,
                               env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
            except Exception:
                shutil.rmtree(target, ignore_errors=True)
        dropped = target.parent / BUNDLE_NAME   # /content/studio_bundle.zip, dragged into the Files panel
        if not (target / "src" / "studio_ui.py").is_file() and dropped.is_file():
            print("Could not download from GitHub — using", BUNDLE_NAME, "from the Files panel.")
            try:
                _install_bundle(dropped.read_bytes(), target)
            except Exception as e:  # noqa: BLE001
                print("Could not use", BUNDLE_NAME, "from the Files panel:", e)
        if (target / "src" / "studio_ui.py").is_file():
            studio = _boot(target)
        else:
            _ask_for_bundle(target)
        return
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "studio_ui.py").is_file():
            studio = _boot(cand)
            return
    print("Could not find the studio files. Set STUDIO_REPO_ROOT or open this notebook inside the repository.")


if not os.environ.get("STUDIO_SETUP_DEFINE_ONLY"):
    _setup()


---
## 1 · Connect (≈5 min)
1. You start in **Replay Mode** — prepared, simulated outputs (no key, no cost). The orange badge in the status bar always tells you which mode you are in.
2. To use a live AI: choose **Provider** and **Model**, add your key to **Colab Secrets** (the box below shows the exact secret name), click **Load my key**, then **Test connection**.
3. Anything not working? Click the big **Use Replay Mode instead** button and carry on — nothing is lost.

*Models marked “not yet verified” have not been tested for this course.*

In [ ]:
#@title ▶ Show Section 1 · Connect { display-mode: "form" }
studio.section(1)

---
## 2 · Choose your mission (≈5 min)
1. Pick **Mission 1 (Introductory)** first. The other missions are for *after the intro* (≈ 35–40 min each; skip Section 1 if you are still connected).
2. Choose the **English variant** your players expect.
3. Read the source strings and open **Scene**, **Character profiles**, **Intended meaning** and **Glossary**. Required glossary terms are flagged in red.
4. Click **Start this mission**.

*Optional:* upload your own text (.txt / .csv / .json). Custom material can be translated and exported, but it cannot be played in the game.

In [ ]:
#@title ▶ Show Section 2 · Choose your mission { display-mode: "form" }
studio.section(2)

<div style="border:3px solid #d35400;border-radius:8px;padding:10px 14px;background:#fff4e5;color:#1f2933"><b>🔁 Swap roles now.</b> Whoever used the keyboard in Sections 1–2 becomes the <b>Quality Evaluator</b>; the partner becomes the <b>AI Director</b> for Sections 3–4.<br>Change <i>“Who is deciding”</i> in the status bar so every decision is recorded under the right role.</div>

---
## 3 · Configure your agents (≈15 min)
**Translator:** start from a preset — **Faithful**, **Balanced** or **Creative** — then change the sliders and rewrite the instructions in your own words. Each preset only sets these visible controls.

**Reviewer:** choose what it checks, how strict it is and how many suggestions it may make.

Click **Show the full prompt that will be sent**: this is *exactly* what the model receives. Nothing is hidden.

> 💬 *Quality Evaluator:* which line of the prompt do you expect to change the translation most?

*In Replay Mode each preset has a prepared output. Your own edits are kept in the prompt and the report, but only Live Mode can generate a new response from them.*

In [ ]:
#@title ▶ Show Section 3 · Configure your agents { display-mode: "form" }
studio.section(3)

---
## 4 · Run the agents (≈20 min)
1. **1 · Baseline (one-shot)** — a deliberately minimal prompt, to see what the agents add.
2. **2 · Run translator agent** — uses your configuration. The button shows how many requests it will make.
3. ⭐ **The key activity:** go back to Section 3, **change the translator's prompt** (another preset, a slider, your own instruction) and click **2 · Run translator agent** again. Every run becomes a new version.
4. **Compare runs:** pick two versions — changed strings are highlighted and the prompt differences are shown underneath. *What in the prompt caused these differences?*
5. **3 · Run reviewer agent** — its findings are **suggestions: you decide**. Automatic technical checks are shown separately.

In [ ]:
#@title ▶ Show Section 4 · Run the agents { display-mode: "form" }
studio.section(4)

<div style="border:3px solid #d35400;border-radius:8px;padding:10px 14px;background:#fff4e5;color:#1f2933"><b>🔁 Swap roles now.</b> The <b>AI Director</b> and the <b>Quality Evaluator</b> change places for the Human Director step.<br>Change <i>“Who is deciding”</i> in the status bar so every decision is recorded under the right role.</div>

---
## 5 · Human Director (≈25 min — the most important part)
For **every** reviewer suggestion choose: **Accept**, **Reject**, **Accept with my edit**, or **Override** with your own wording — and write *why* in the comment. Some suggestions are wrong or debatable — on purpose!

Then:
* **Add a project rule** for anything the agents should always follow (project memory — the model itself is not retrained).
* **Send accepted feedback to the translator** (only those strings are re-sent; your own wording is applied directly).
* Optionally **review the revised strings**, edit any string yourself, compare or restore versions.
* **Approve final translation** when you are satisfied. Any later change revokes the approval.
* Then click **Download progress** in the status bar: the autosave is lost if Colab resets the runtime.

In [ ]:
#@title ▶ Show Section 5 · Human Director { display-mode: "form" }
studio.section(5)

---
## 6 · Test and export (≈10 min)
1. Check the summary: Source · Baseline · Agent-reviewed · Final approved.
2. Answer the reflection question together and click **Save reflection**.
3. Export **game_translation_en.json**, **agent_settings.json** and **workflow_report.html** (files are also saved in *exports → your mission*, e.g. `exports/intro/` — folder icon in Colab's sidebar). Every mission uses the same file names, so your browser may add “(1)” to a second download.
4. **Play it:** open the game → **Import student JSON**. The game switches **Dialogue language** to **Student's imported English** by itself.

Finished early? Click **Start the next mission** (your finished project is kept as a file).

In [ ]:
#@title ▶ Show Section 6 · Test and export { display-mode: "form" }
studio.section(6)

<div style="border:3px solid #d35400;border-radius:8px;padding:10px 14px;background:#fff4e5;color:#1f2933"><b>🔁 Swap roles now.</b> Before the next mission, swap roles again, then scroll back up to Section 2 — your new mission is already selected.<br>Change <i>“Who is deciding”</i> in the status bar so every decision is recorded under the right role.</div>

---
## 🆘 If you get stuck
* **Colab disconnected?** Run Step 0 again, then **Resume my saved session** (works while the runtime still exists).
* **Colab reset the runtime?** The autosave is gone: choose your downloaded progress file and click **Resume from file**.
* **Behind the class?** Choose a **Checkpoint:** (“Catch up: …”) and click **Load this checkpoint**.
* **New computer?** **Download progress file** now; later use **Resume from file**.
* **Mission in a mess?** **Reset this mission** (asks first; a backup is kept).

In [ ]:
#@title ▶ Show the help box { display-mode: "form" }
studio.stuck()